# Constrovet Gemini Executive Verifier

Optional no-GCP verifier for the active Constrovet browser dashboard.

Use this notebook after running `https://www.constrovet.com/app/` and downloading `executive_synthesis.json`. The notebook sends only cited findings, calculations, action plans, honesty check, and model audit metadata to Gemini. It does not upload raw project documents, does not use Cloud Run, does not use GCS, and does not require a service account.

Expected Drive layout:

```text
My Drive/Constrovet/projects/<project_id>/outputs/executive_synthesis.json
My Drive/Constrovet/projects/<project_id>/outputs/gemini_executive_verifier.json
My Drive/Constrovet/projects/<project_id>/outputs/gemini_executive_verifier.md
```

Budget guardrail: use Colab free runtime and the already-approved Gemini API/subscription only.


In [ ]:
#@title 1. Install dependency
!pip -q install google-genai


In [ ]:
#@title 2. Load verifier code
from __future__ import annotations

import datetime as _dt
import json
import os
import re
from pathlib import Path
from typing import Any

try:
    from google.colab import drive, files, userdata  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    files = None
    userdata = None
    IN_COLAB = False

PROJECT_ID = os.getenv("CONSTROVET_PROJECT_ID", "sample-project")
DRIVE_ROOT = Path("/content/drive/MyDrive") if IN_COLAB else Path.cwd()
PROJECT_ROOT = DRIVE_ROOT / "Constrovet" / "projects" / PROJECT_ID
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
BROWSER_JSON_PATH = OUTPUTS_DIR / "executive_synthesis.json"
VERIFIER_JSON_PATH = OUTPUTS_DIR / "gemini_executive_verifier.json"
VERIFIER_MD_PATH = OUTPUTS_DIR / "gemini_executive_verifier.md"
GEMINI_MODEL = os.getenv("GEMINI_VERIFIER_MODEL", "gemini-2.5-pro")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
if not GEMINI_API_KEY and userdata is not None:
    try:
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY") or ""
    except Exception:
        GEMINI_API_KEY = ""

STRICT_VERIFIER_PROMPT = """
You are Constrovet's evidence-bound executive verifier for construction project cost intelligence.

INPUT POLICY:
- You will receive only structured browser findings, cited spans, calculations, action plans, honesty check, and model audit metadata.
- Do not ask for or infer from raw documents.
- Do not invent amounts, dates, causes, entitlement, recovery probability, party fault, legal conclusions, or certification language.

EVIDENCE RULES:
1. Use only cited findings and quoted spans provided in the input.
2. If Budget and Actual are both available, overrun is Actual - Budget.
3. If Actual > Budget, the finding may support LEAKAGE_AND_OVERRUN.
4. Baseline BOQ, contract value, planned spend, cumulative work done, and advance recovery are not leakage.
5. ESG metrics remain ESG_METRIC unless separate financial evidence supports a cost finding.
6. If evidence is weak or missing, put the issue in honesty_check and unsupported_claims_removed.

TASK:
Rewrite and verify the browser output for senior executives. Keep it concise, commercially useful, and evidence-bound.

RETURN ONLY STRICT JSON WITH THIS SHAPE:
{
  "executive_brief": ["..."],
  "board_summary": ["..."],
  "top_decisions_required": ["..."],
  "contractor_questions": ["..."],
  "recovery_actions": [
    {
      "action": "...",
      "why_now": "...",
      "source_finding_indexes": [1],
      "expected_evidence_to_collect": ["..."],
      "confidence": "HIGH | MEDIUM | LOW"
    }
  ],
  "verification_status": "VERIFIED_WITH_LIMITATIONS | NEEDS_REVIEW | REJECTED_UNSUPPORTED",
  "unsupported_claims_removed": ["..."],
  "honesty_check": {
    "missing_evidence": ["..."],
    "assumptions": ["..."]
  },
  "model_audit_trail": {
    "model": "...",
    "input_policy": "cited findings only",
    "raw_documents_sent": false,
    "generated_at": "ISO-8601 timestamp"
  }
}
""".strip()

ALLOWED_FINDING_KEYS = {
    "statement", "financial_category", "amount_inr", "days", "citations", "calculation", "confidence",
    "risk_score", "severity", "recoverability", "evidence_quality", "cash_impact", "urgency", "control_theme"
}

ALLOWED_TOP_LEVEL_KEYS = {
    "findings", "executive_brief", "top_5_actions", "recoverable_cost_exposure", "immediate_control_failures",
    "missing_evidence_blocking_recovery", "executive_action_plan", "rationale", "model_audit_trail", "honesty_check",
    "gemini_verifier_result"
}

def load_browser_report(path: Path = BROWSER_JSON_PATH) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Browser JSON not found: {path}")
    return json.loads(path.read_text(encoding="utf-8"))

def validate_browser_report(report: dict[str, Any]) -> None:
    if not isinstance(report.get("findings"), list):
        raise ValueError("executive_synthesis.json must contain findings[].")
    for idx, finding in enumerate(report["findings"], 1):
        citations = finding.get("citations") or []
        if not citations:
            raise ValueError(f"Finding {idx} has no citation.")
        citation = citations[0]
        if not citation.get("file") or not citation.get("page_or_sheet") or not citation.get("quoted_span"):
            raise ValueError(f"Finding {idx} citation must include file, page_or_sheet, and quoted_span.")
        if finding.get("financial_category") not in {"BASELINE_BUDGET", "LEAKAGE_AND_OVERRUN", "ESG_METRIC"}:
            raise ValueError(f"Finding {idx} has invalid financial_category.")
        calc = finding.get("calculation") or {}
        budget = float(calc.get("budget") or 0)
        actual = float(calc.get("actual") or 0)
        difference = float(calc.get("difference") or 0)
        if budget and actual and actual > budget and abs((actual - budget) - difference) > 0.01:
            raise ValueError(f"Finding {idx} has inconsistent Actual - Budget calculation.")

def _trim_text(value: Any, limit: int = 900) -> Any:
    if isinstance(value, str):
        text = re.sub(r"\s+", " ", value).strip()
        return text[:limit]
    if isinstance(value, list):
        return [_trim_text(item, limit) for item in value]
    if isinstance(value, dict):
        return {key: _trim_text(item, limit) for key, item in value.items()}
    return value

def sanitize_for_gemini(report: dict[str, Any]) -> dict[str, Any]:
    validate_browser_report(report)
    safe: dict[str, Any] = {key: _trim_text(report[key]) for key in ALLOWED_TOP_LEVEL_KEYS if key in report and key != "findings"}
    safe["findings"] = []
    for finding in report.get("findings", []):
        safe_finding = {key: _trim_text(finding.get(key)) for key in ALLOWED_FINDING_KEYS if key in finding}
        safe["findings"].append(safe_finding)
    safe["input_guardrail"] = {
        "raw_documents_sent": False,
        "source": "browser executive_synthesis.json",
        "allowed_content": "findings, citations, calculations, action plan, honesty check, and audit metadata only"
    }
    return safe

def build_prompt(safe_payload: dict[str, Any]) -> str:
    return STRICT_VERIFIER_PROMPT + "\n\nBROWSER_OUTPUT_JSON:\n" + json.dumps(safe_payload, ensure_ascii=False, indent=2)

def extract_json(text: str) -> dict[str, Any]:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError("Gemini response did not contain a JSON object.")
    return json.loads(text[start:end + 1])

def call_gemini_verifier(safe_payload: dict[str, Any], model: str = GEMINI_MODEL) -> dict[str, Any]:
    if not GEMINI_API_KEY:
        raise RuntimeError("GEMINI_API_KEY is required. Add it as a Colab secret or environment variable.")
    from google import genai
    try:
        from google.genai import types
    except Exception:
        types = None

    client = genai.Client(api_key=GEMINI_API_KEY)
    prompt = build_prompt(safe_payload)
    if types is not None:
        response = client.models.generate_content(
            model=model,
            contents=prompt,
            config=types.GenerateContentConfig(response_mime_type="application/json")
        )
    else:
        response = client.models.generate_content(model=model, contents=prompt)
    result = extract_json(response.text or "")
    result.setdefault("model_audit_trail", {})
    result["model_audit_trail"].update({
        "model": model,
        "input_policy": "cited findings only",
        "raw_documents_sent": False,
        "generated_at": _dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
    })
    return result

def fallback_verifier(report: dict[str, Any]) -> dict[str, Any]:
    safe = sanitize_for_gemini(report)
    findings = safe.get("findings", [])
    leakage = [item for item in findings if item.get("financial_category") == "LEAKAGE_AND_OVERRUN"]
    return {
        "executive_brief": [safe.get("executive_brief", {}).get("headline", "No executive brief was provided by the browser output.")],
        "board_summary": [
            f"Browser triage produced {len(findings)} cited finding(s), including {len(leakage)} leakage/overrun finding(s).",
            "Gemini was not run; this is a deterministic fallback summary."
        ],
        "top_decisions_required": [item.get("action", "Review cited finding") for item in safe.get("top_5_actions", [])[:5]],
        "contractor_questions": [
            "Which cited invoice, BOQ, RA bill, IPC, delay, or ESG record supports each finding?",
            "What missing approval, reconciliation, or site record is needed before recovery action?"
        ],
        "recovery_actions": [
            {
                "action": item.get("action", "Review cited finding"),
                "why_now": "Browser risk scoring marked this as a priority cited action.",
                "source_finding_indexes": [item.get("source_finding_index")],
                "expected_evidence_to_collect": ["invoice/RA bill support", "BOQ/budget line", "approval trail", "site record"],
                "confidence": "MEDIUM"
            }
            for item in safe.get("top_5_actions", [])[:5]
        ],
        "verification_status": "NEEDS_REVIEW",
        "unsupported_claims_removed": ["Gemini was not run, so no LLM-supported wording verification was performed."],
        "honesty_check": {
            "missing_evidence": safe.get("missing_evidence_blocking_recovery", []),
            "assumptions": ["Fallback verifier used deterministic browser output only."]
        },
        "model_audit_trail": {
            "model": "none",
            "input_policy": "cited findings only",
            "raw_documents_sent": False,
            "generated_at": _dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
        }
    }

def write_markdown(result: dict[str, Any], path: Path = VERIFIER_MD_PATH) -> None:
    lines = ["# Constrovet Gemini Executive Verifier", ""]
    for title, key in [
        ("Executive Brief", "executive_brief"),
        ("Board Summary", "board_summary"),
        ("Top Decisions Required", "top_decisions_required"),
        ("Contractor Questions", "contractor_questions"),
        ("Unsupported Claims Removed", "unsupported_claims_removed"),
    ]:
        lines += [f"## {title}", ""]
        values = result.get(key) or []
        if values:
            lines += [f"- {item}" for item in values]
        else:
            lines.append("- None recorded.")
        lines.append("")
    lines += ["## Recovery Actions", ""]
    for item in result.get("recovery_actions", []) or []:
        lines.append(f"- {item.get('action', 'Action')}")
        lines.append(f"  - Why now: {item.get('why_now', '')}")
        lines.append(f"  - Source findings: {', '.join(str(x) for x in item.get('source_finding_indexes', []) if x is not None) or 'None'}")
        lines.append(f"  - Evidence to collect: {', '.join(item.get('expected_evidence_to_collect', []))}")
        lines.append(f"  - Confidence: {item.get('confidence', '')}")
    lines += ["", "## Honesty Check", ""]
    honesty = result.get("honesty_check", {}) or {}
    for item in honesty.get("missing_evidence", []) or []:
        lines.append(f"- Missing evidence: {item}")
    for item in honesty.get("assumptions", []) or []:
        lines.append(f"- Assumption: {item}")
    audit = result.get("model_audit_trail", {}) or {}
    lines += ["", "## Model Audit Trail", ""]
    for key, value in audit.items():
        lines.append(f"- {key}: {value}")
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")

def write_outputs(result: dict[str, Any]) -> None:
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    VERIFIER_JSON_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    write_markdown(result)
    print("Wrote", VERIFIER_JSON_PATH)
    print("Wrote", VERIFIER_MD_PATH)

def run_verifier(use_gemini: bool = True) -> dict[str, Any]:
    report = load_browser_report()
    safe_payload = sanitize_for_gemini(report)
    if use_gemini:
        result = call_gemini_verifier(safe_payload)
    else:
        result = fallback_verifier(report)
    write_outputs(result)
    return result

def run_smoke_tests() -> None:
    sample = {
        "findings": [{
            "statement": "CSV row shows Actual - Budget overrun of INR 50.",
            "financial_category": "LEAKAGE_AND_OVERRUN",
            "amount_inr": 50,
            "days": 0,
            "citations": [{"file": "smoke.csv", "page_or_sheet": "CSV row 2", "quoted_span": "Budget: INR 100 | Actual: INR 150"}],
            "calculation": {"budget": 100, "actual": 150, "difference": 50, "formula": "Actual - Budget"},
            "confidence": "HIGH",
            "risk_score": 65,
            "severity": "HIGH",
            "recoverability": "HIGH",
            "evidence_quality": "STRUCTURED_ACTUAL_BUDGET"
        }],
        "executive_brief": {"headline": "Smoke overrun", "decision_focus": "Finding 1"},
        "top_5_actions": [{"action": "Validate INR 50 overrun", "source_finding_index": 1}],
        "missing_evidence_blocking_recovery": [],
        "executive_action_plan": {},
        "honesty_check": {"missing_evidence": [], "documents_not_processed": [], "assumptions": []}
    }
    validate_browser_report(sample)
    safe = sanitize_for_gemini(sample)
    prompt = build_prompt(safe)
    assert "raw_documents_sent" in prompt
    assert "Budget: INR 100" in prompt
    fallback = fallback_verifier(sample)
    assert fallback["verification_status"] == "NEEDS_REVIEW"
    assert fallback["model_audit_trail"]["raw_documents_sent"] is False
    print("Smoke tests passed without Gemini API call.")

print("Loaded Constrovet Gemini verifier.")
print("Project ID:", PROJECT_ID)
print("Expected browser JSON:", BROWSER_JSON_PATH)
print("Gemini model:", GEMINI_MODEL)


In [ ]:
#@title 3. Mount Drive and locate browser JSON
if IN_COLAB:
    drive.mount('/content/drive')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print('Project ID:', PROJECT_ID)
print('Outputs folder:', OUTPUTS_DIR)
print('Expected browser JSON:', BROWSER_JSON_PATH)
if BROWSER_JSON_PATH.exists():
    print('Found browser JSON.')
elif IN_COLAB and files is not None:
    print('Browser JSON not found in Drive. Upload executive_synthesis.json now if needed.')
    uploaded = files.upload()
    for name, data in uploaded.items():
        if name.endswith('.json'):
            BROWSER_JSON_PATH.write_bytes(data)
            print('Saved uploaded JSON to', BROWSER_JSON_PATH)
            break
else:
    print('Browser JSON not found. Place executive_synthesis.json in the outputs folder.')


In [ ]:
#@title 4. Run optional Gemini verifier
# Requires GEMINI_API_KEY as a Colab secret or environment variable.
# Set use_gemini=False for a no-API deterministic fallback smoke run.
# result = run_verifier(use_gemini=True)


In [ ]:
#@title 5. Smoke tests - no Gemini API call
run_smoke_tests()
